In [1]:
import sys
import numpy as np
import pybullet as p
import time
import logging
from typing import List, Tuple, Optional, Sequence, Collection, Dict, Any, cast
import random
import json

from predicators.structs import Action, Array, GroundAtom, Object, State, Type, ParameterizedOption
from predicators import utils
from predicators.settings import CFG
from gym.spaces import Box

#Import core environment methods, robot function etc.

from predicators.envs.pybullet_blocks import PyBulletBlocksEnv
from predicators.envs.pybullet_env import PyBulletEnv, create_pybullet_block
from predicators.envs.pybullet_multitable_blocks import PyBulletMultiTableBlocksEnv
from predicators.pybullet_helpers.robots import SingleArmPyBulletRobot
from predicators.pybullet_helpers.robots.mobile_single_arm import MobileSingleArmPyBulletRobot
from predicators.pybullet_helpers.geometry import Pose
from predicators.pybullet_helpers.joint import JointPositions, get_joint_infos, get_joint_positions
from predicators.pybullet_helpers.link import get_link_state, get_link_pose

pybullet build time: Jan 29 2025 23:16:28


In [2]:
logging.basicConfig(
    level=logging.WARNING,                    
    format="%(asctime)s %(name)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

#Defining test configuration, and overriding some default ones:
CFG.pybullet_robot = "fetch_mobile"
CFG.use_gui = True
CFG.env = "pybullet_multitable_blocks"
#Draws helpful debug lines in the workspace.
#NOT SURE WHETHER TO USE THIS. WILL DECIDE AFTER A COUPLE RUNS.
#CFG.pybullet_draw_debug = True
#Initializing with standard size of blocks.
CFG.blocks_block_size = 0.05
CFG.pybullet_birrt_num_iters = 50
CFG.pybullet_birrt_num_attempts = 10
CFG.pybullet_birrt_smooth_amt = 20
CFG.seed = random.randint(0,10000)
#CFG.seed = 12
#Num of PyBullet physics steps per high-level Action in visualize_action_sequence
CFG.pybullet_sim_steps_per_action = 200

In [3]:
multitable_env = PyBulletMultiTableBlocksEnv(use_gui=True)

In [4]:
table_configs = {  
    0: {  # Table 0: Random piles  
        'exact_state': {},  # Not used for 'pile' mode  
        'setup': 'pile',  
        'params': [2, 3]  # 2 piles, 3 blocks per pile  
    },  
    1: {  # Table 1: Exact pile configuration  
        'exact_state': {  
            'pile1': ['red', 'blue', 'green'],  
            'pile2': ['yellow', 'purple'],  
            'pile3': ['orange']  
        },  
        'setup': 'exact_pile',  
        'params': None  
    },  
    2: {  # Table 2: Scattered blocks  
        'exact_state': ['cyan', 'magenta', 'lime', 'pink'],  
        'setup': 'exact_scattered',  
        'params': None  
    }  
}

In [5]:
multitable_env.set_state(table_configs)

Config for table 2:{'exact_state': ['cyan', 'magenta', 'lime', 'pink'], 'setup': 'exact_scattered', 'params': None}.


Created pile: [[block2_1_0:block], [block2_2_1:block], [block2_3_2:block], [block2_4_3:block]].


Table state returned for 3rd table:{block2_1_0:block: array([2.23501144e+00, 1.92923685e+00, 2.25000000e-01, 0.00000000e+00,
       0.00000000e+00, 2.55000000e+02, 2.55000000e+02]), block2_2_1:block: array([2.29895022e+00, 2.19357812e+00, 2.25000000e-01, 0.00000000e+00,
       2.55000000e+02, 0.00000000e+00, 2.55000000e+02]), block2_3_2:block: array([2.41909263e+00, 1.96469090e+00, 2.25000000e-01, 0.00000000e+00,
       0.00000000e+00, 2.55000000e+02, 0.00000000e+00]), block2_4_3:block: array([2.29538791e+00, 2.07637060e+00, 2.25000000e-01, 0.00000000e+00,
       2.55000000e+02, 1.92000000e+02, 2.03000000e+02])}.


PyBulletState(data={table0:table: array([ 1. , -0.5,  0. ,  0. ], dtype=float32), table1:table: array([-1.7,  1.5,  0. ,  1. ], dtype=float32), table2:table: array([2.35, 2.  , 0.  , 2.  ], dtype=float32), robby:robot: array([0.55, 1.  , 0.01, 1.  ], dtype=float32), block0_0_0:block: array([ 1.05536373, -0.64552698,  0.225     ,  0.        ,  0.30583298,
        0.78233463,  0.07305057]), block0_0_1:block: array([ 1.05536373, -0.64552698,  0.275     ,  0.        ,  0.83672925,
        0.85590298,  0.21462729]), block0_0_2:block: array([ 1.05536373, -0.64552698,  0.325     ,  0.        ,  0.75084281,
        0.67511398,  0.87304587]), block0_1_0:block: array([ 1.08871922, -0.49498472,  0.225     ,  0.        ,  0.31338413,
        0.52143128,  0.09022519]), block0_1_1:block: array([ 1.08871922, -0.49498472,  0.275     ,  0.        ,  0.54299081,
        0.9773304 ,  0.73961831]), block0_1_2:block: array([ 1.08871922, -0.49498472,  0.325     ,  0.        ,  0.36980457,
        0.72821837